# Week 7：XGBoost 入门

目标：使用 XGBoost 完成一个结构化分类任务的 5 折交叉验证，并理解最常用的复杂度与随机性参数。当前要求是会用、会解释，不要求推导完整目标函数。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from xgboost import XGBClassifier

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 1. 一组保守的起始参数

- `n_estimators`：树的数量。
- `learning_rate`：每棵树的修正步长。
- `max_depth`：每棵树的最大深度。
- `subsample`：每棵树随机使用的训练样本比例。
- `colsample_bytree`：每棵树随机使用的特征比例。
- `reg_lambda`：L2 正则化强度，帮助控制模型复杂度。

这些参数不是记忆题；重点是知道它们分别控制树数量、单次修正幅度、树复杂度、样本随机性、特征随机性和正则化。

In [3]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
)

scores = cross_validate(
    xgb_model,
    X_train,
    y_train,
    cv=cv,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1,
)

xgb_result = pd.DataFrame([{
    'train_mean': scores['train_score'].mean(),
    'validation_mean': scores['test_score'].mean(),
    'validation_std': scores['test_score'].std(),
    'fold_scores': scores['test_score'],
}])
display(xgb_result)

,train_mean,validation_mean,validation_std,fold_scores
0,1.0,0.973626,0.016447,"[0.989010989010989, 0.978021978021978, 0.94505..."


## 检查点

1. `subsample=0.8` 与 `colsample_bytree=0.8` 分别随机了什么？
2. 为什么 `learning_rate=0.05` 通常需要配合较多树？
3. 若训练平均准确率接近 1，而交叉验证平均准确率明显更低，你会优先检查哪两个复杂度参数？